# Richmond OSM + Overture Walkthrough

This notebook is a quick review aid for two decisions:

1. whether `pyrosm` is the right package for Richmond-scale OSM infrastructure ingestion
2. what the landed Richmond Overture POI data looks like, how it is shaped, and how we should classify it


## Part 1 — How `pyrosm` works

`pyrosm` works from local or downloaded `.osm.pbf` extracts, not from live Overpass queries. That is why it is attractive for metro-scale infrastructure work.

### OSM data model

OSM's native grain is not a ready-made GIS feature table. It is built from:
- **nodes**: points with coordinates
- **ways**: ordered node sequences used for lines or polygon boundaries
- **relations**: grouped objects like multipolygons or route structures

`pyrosm` reads a `.pbf` snapshot and reconstructs feature tables from those OSM elements.

### What `pyrosm` returns

- `OSM.get_network(...)` returns a GeoDataFrame of **edges** by default.
- `OSM.get_network(..., nodes=True)` returns **(nodes, edges)** so the same street data can become a graph.
- `OSM.get_pois()` returns POI features as a GeoDataFrame.
- `OSM.get_buildings()`, `get_landuse()`, `get_boundaries()`, and `get_data_by_custom_criteria()` give the same style of GeoDataFrame output for other themes.

So yes, it absolutely can work at the **street-network level**. That is one of the main reasons to use it.

### Practical grains to expect in our workflow

- **Street network edges**: one row per road segment / parsed edge
- **Street network nodes**: one row per node / intersection point when requested
- **POIs**: one row per parsed OSM POI feature
- **Buildings / polygons**: one row per parsed building or area feature

For D4, the useful first-pass outputs would likely be:
- road / highway edges
- rail edges
- airport / port / logistics points or polygons where present


## `pyrosm` example workflow for Richmond

This is the intended pattern for a Richmond test. It is shown as code, not yet run here.


In [ ]:
# Example only: intended pyrosm workflow for Richmond
from pyrosm import OSM

# 1. Work from a downloaded Virginia or Richmond-area PBF
pbf_path = '/path/to/virginia-latest.osm.pbf'

# 2. Use a Richmond bbox or polygon footprint to limit parsing
richmond_bbox = [-78.290703, 36.657218, -76.595644, 38.058168]
osm = OSM(pbf_path, bounding_box=richmond_bbox)

# 3. Get a road network at street-segment grain
edges = osm.get_network(network_type='driving')

# 4. If we want a graph-ready street network
nodes, edges_for_graph = osm.get_network(network_type='driving', nodes=True)

# 5. If we want custom rail or other infrastructure filters
rail = osm.get_data_by_custom_criteria(
    custom_filter={'railway': ['rail', 'light_rail', 'subway']},
    filter_type='keep',
    keep_nodes=False,
    keep_ways=True,
    keep_relations=False,
)

edges.head(), rail.head()

## Why `pyrosm` is a strong fit for us

- It avoids live Overpass fragility for full-metro runs.
- It can return real network edges and nodes, not just generalized place rows.
- It lets us ingest once from a `.pbf` snapshot and then derive multiple infrastructure layers locally.
- It fits the Richmond-first pattern: download once, inspect repeatedly, refine filters without re-querying a public API.


## Part 2 — Richmond Overture POI data

Below we inspect the Richmond metro Overture slice that already landed locally.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != 'industry':
    ROOT = ROOT / 'metro-deep-dive' / 'metro-area-explorer' / 'industry'

OUTPUT_DIR = ROOT / 'outputs' / 'richmond_va'
OVERTURE_PATH = OUTPUT_DIR / 'overture_pois.parquet'
MANIFEST_PATH = OUTPUT_DIR / 'spatial_manifest.json'

con = duckdb.connect()
OVERTURE_PATH

PosixPath('/Users/danberle/Documents/projects/patterns_in_place/metro-deep-dive/metro-area-explorer/industry/outputs/richmond_va/overture_pois.parquet')

## Landed shape

This is the app/cache-oriented shape we currently store.

In [2]:
sample = con.execute('SELECT * FROM read_parquet(?) LIMIT 5', [str(OVERTURE_PATH)]).fetchdf()
sample

,market_id,source_system,source_id,feature_name,layer_group,category,subcategory,geometry_type,geometry,centroid_lat,centroid_lon,attributes_json,extract_date
0,40060,overture,c715291a-7006-4fb2-99af-412c74df26ae,Prince Edward State Forest,overture_pois,poi,park,Point,"{""type"": ""Point"", ""coordinates"": [-78.28469706...",37.161512,-78.284697,"{""address_freeform"": null, ""basic_category"": ""...",2026-07-29T00:31:54+00:00
1,40060,overture,30b9779a-e53a-4802-9222-8aad0deee743,White Swan Lodge,overture_pois,poi,lodging,Point,"{""type"": ""Point"", ""coordinates"": [-78.28470406...",37.161390,-78.284704,"{""address_freeform"": "" "", ""basic_category"": ""l...",2026-07-29T00:31:54+00:00
2,40060,overture,fe1b2271-5843-4e7b-863f-6b13c6f2e89d,Joyful Earth Farm,overture_pois,poi,farm,Point,"{""type"": ""Point"", ""coordinates"": [-78.28835333...",37.352684,-78.288353,"{""address_freeform"": ""73 High Hill Rd"", ""basic...",2026-07-29T00:31:54+00:00
3,40060,overture,0561e78a-cff8-4144-a20e-83a5a38adeb5,Jamestown Presbyterian Church,overture_pois,poi,christian_place_of_worship,Point,"{""type"": ""Point"", ""coordinates"": [-78.28802533...",37.298514,-78.288025,"{""address_freeform"": ""1751 Lockett Rd"", ""basic...",2026-07-29T00:31:54+00:00
4,40060,overture,dea2a94f-b815-4420-90f0-f0c9fb62ed7b,Mam Jam's Candle Company,overture_pois,poi,hardware_home_and_garden_store,Point,"{""type"": ""Point"", ""coordinates"": [-78.2876889,...",37.300141,-78.287689,"{""address_freeform"": ""1876 Lockett Rd"", ""basic...",2026-07-29T00:31:54+00:00


In [3]:
sample.columns.tolist()

['market_id',
 'source_system',
 'source_id',
 'feature_name',
 'layer_group',
 'category',
 'subcategory',
 'geometry_type',
 'geometry',
 'centroid_lat',
 'centroid_lon',
 'attributes_json',
 'extract_date']

### What this shape means

Current stored columns:
- `market_id`
- `source_system`
- `source_id`
- `feature_name`
- `layer_group`
- `category`
- `subcategory`
- `geometry_type`
- `geometry`
- `centroid_lat`
- `centroid_lon`
- `attributes_json`
- `extract_date`

The important design choice is that we keep both:
- a **modeled** category/subcategory
- the **raw source metadata** inside `attributes_json`

That lets us re-classify later without re-ingesting.

## What raw Overture metadata we preserved

Inside `attributes_json` we currently keep the category fields needed for comparison:
- legacy `primary_category`
- newer `basic_category`
- newer `taxonomy_primary`
- `taxonomy_hierarchy`
- `confidence`
- `address_freeform`


In [4]:
con.execute("""
SELECT
  feature_name,
  json_extract_string(attributes_json, '$.primary_category') AS primary_category,
  json_extract_string(attributes_json, '$.basic_category') AS basic_category,
  json_extract_string(attributes_json, '$.taxonomy_primary') AS taxonomy_primary,
  json_extract(attributes_json, '$.taxonomy_hierarchy') AS taxonomy_hierarchy,
  json_extract(attributes_json, '$.confidence') AS confidence,
  json_extract_string(attributes_json, '$.address_freeform') AS address_freeform
FROM read_parquet(?)
LIMIT 10
""", [str(OVERTURE_PATH)]).fetchdf()

,feature_name,primary_category,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,address_freeform
0,Prince Edward State Forest,park,park,park,"[""sports_and_recreation"",""park""]",0.7296755313873291,None
1,White Swan Lodge,lodge,lodging,lodge,"[""lodging"",""lodge""]",0.9105160236358643,
2,Joyful Earth Farm,farm,farm,farm,"[""services_and_business"",""agricultural_service...",0.6143476963043213,73 High Hill Rd
3,Jamestown Presbyterian Church,church_cathedral,christian_place_of_worship,christian_place_of_worship,"[""cultural_and_historic"",""religious_organizati...",0.9199122190475464,1751 Lockett Rd
4,Mam Jam's Candle Company,candle_store,hardware_home_and_garden_store,candle_store,"[""shopping"",""specialty_store"",""hardware_home_a...",0.9199122190475464,1876 Lockett Rd
5,Pisgah Baptist Church,baptist_church,christian_place_of_worship,baptist_place_of_worship,"[""cultural_and_historic"",""religious_organizati...",0.9199122190475464,202 Pisgah Church Rd
6,Rice Fire House,fire_department,fire_station,fire_station,"[""community_and_government"",""public_safety_ser...",0.9199122190475464,948 Rices Depot Rd
7,Dollar General,discount_store,discount_store,discount_store,"[""shopping"",""discount_store""]",0.9954287385940552,1001 Rices Depot Rd
8,Western Union,money_transfer_services,financial_service,money_transfer_service,"[""services_and_business"",""financial_service"",""...",0.9199122190475464,1001 Rices Depot Rd
9,The Farmer's Daughters,farm,farm,farm,"[""services_and_business"",""agricultural_service...",0.9199122190475464,24120 Prince Edward Hwy


## Summary stats

### Total landed Richmond POIs

In [5]:
con.execute('SELECT count(*) AS total_rows FROM read_parquet(?)', [str(OVERTURE_PATH)]).fetchdf()

,total_rows
0,76913


### Top modeled subcategories

In [6]:
con.execute("""
SELECT category, subcategory, count(*) AS rows
FROM read_parquet(?)
GROUP BY 1,2
ORDER BY rows DESC
LIMIT 20
""", [str(OVERTURE_PATH)]).fetchdf()

,category,subcategory,rows
0,poi,home_service,4153
1,poi,restaurant,4088
2,poi,place,3168
3,poi,financial_service,3102
4,poi,personal_or_beauty_service,3048
5,poi,christian_place_of_worship,2997
6,poi,real_estate_service,2831
7,infrastructure,port,2155
8,poi,automotive_service,2126
9,poi,hardware_home_and_garden_store,1900


### First-wave amenity counts

In [7]:
con.execute("""
SELECT subcategory, count(*) AS rows
FROM read_parquet(?)
WHERE category = 'amenity'
GROUP BY 1
ORDER BY rows DESC
""", [str(OVERTURE_PATH)]).fetchdf()

,subcategory,rows
0,grocery,999
1,hospital,255


## Grocery count caveat

The current `999` grocery-like count is **not** a system cap, but it is **not yet a trustworthy grocery count** either.

Why:
- there is no obvious ingestion ceiling because nearby categories also exceed or approach 1000
- the current grocery mapping is too loose and is catching categories that merely contain `market` in a non-food sense


In [8]:
con.execute("""
SELECT
  json_extract_string(attributes_json, '$.primary_category') AS primary_category,
  count(*) AS rows
FROM read_parquet(?)
WHERE subcategory = 'grocery'
GROUP BY 1
ORDER BY rows DESC
LIMIT 25
""", [str(OVERTURE_PATH)]).fetchdf()

,primary_category,rows
0,grocery_store,424
1,farmers_market,134
2,supermarket,128
3,marketing_agency,123
4,specialty_grocery_store,31
5,business_advertising,30
6,health_food_store,28
7,marketing_consultant,25
8,flea_market,25
9,internet_marketing_service,15


That table is the key diagnostic. It shows real grocery classes like `grocery_store` and `supermarket`, but also false-positive classes like `marketing_agency` and `marketing_consultant`.

So the fix is not re-ingestion. The fix is **better downstream classification**.

## Recommended preparation pattern for Overture

Recommended scalable shape:

1. **Ingest broad Richmond metro slice once**
2. **Preserve raw metadata fields**
3. **Build a classification layer downstream**
4. **Create filtered D4 views from the classified table**

Conceptually:

```text
overture_raw_richmond
    -> overture_classified_richmond
        -> hospitals view
        -> groceries view
        -> airport/port anchors view
        -> broader D4 overlay view
```

That means we should classify using:
- `basic_category` first for broad rollups
- `taxonomy_primary` for detail
- `primary_category` as fallback/debug context
- explicit allowlists for sensitive buckets like grocery


## Recommended next step

- Test Richmond OSM infra with `pyrosm` from a `.pbf` extract rather than live Overpass.
- Keep Overture as broad ingest + downstream classification.
- Tighten the grocery rules before treating food-access counts as publishable.